# CLIP Classification

Rank editable labels for an image, inspect tokenization, and draw the results on the image.

In [ ]:
import torch
from diffusers.utils import load_image
from PIL import ImageDraw, ImageFont
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device).eval()
processor = CLIPProcessor.from_pretrained(model_name)

In [ ]:
image = load_image("http://images.cocodataset.org/val2017/000000039769.jpg").convert("RGB")
labels = ["a cat", "a dog", "a person", "an animal", "a pet"]
inputs = processor(text=labels, images=image, return_tensors="pt", padding=True).to(device)
for label in labels:
    print(f"{label!r} -> {processor.tokenizer.tokenize(label)}")
with torch.inference_mode():
    logits = model(**inputs).logits_per_image[0]
    probabilities = logits.softmax(dim=0).cpu().tolist()
ranked = sorted(zip(labels, probabilities, strict=True), key=lambda row: row[1], reverse=True)
ranked

In [ ]:
overlay = image.copy()
draw = ImageDraw.Draw(overlay, "RGBA")
font = ImageFont.load_default(size=18)
for row, (label, probability) in enumerate(ranked):
    text = f"{probability:6.2%}  {label}"
    y = 12 + row * 25
    box = draw.textbbox((12, y), text, font=font)
    draw.rectangle((box[0]-3, box[1]-2, box[2]+3, box[3]+2), fill=(0, 0, 0, 180))
    draw.text((12, y), text, fill="yellow" if row < 3 else "white", font=font)
overlay

CLIP probabilities are relative to the labels supplied in the cell. They are not calibrated class probabilities or moderation scores.